# 05-1. 구간별 추정과 결빙 영향인자

05의 1~2장입니다. 03·04 모델로 구간(1km)별 노면온도와 결빙관련 노면 확률을 1시간 간격으로 추정하고, 학습 지점 관측자료로 결빙 영향인자와 환경조건(임계값)을 정리합니다. 위험도 기준표와 지도는 05-2에서 만듭니다.

02-2의 구간·지점 변수, 03의 노면온도 모델과 구간 B3, 04-1의 결빙 분류 모델, 02-3의 ERA5-Land와 해기차를 읽습니다. 구간별 추정값, 노면온도 요약, 결빙 가능 조건은 `data/processed/`에 저장해서 05-2에서 씁니다. 영향인자와 환경조건 표는 `data/results/icing_factors.csv`, `icing_conditions.csv`, 구간별 표는 `segment_factors.csv`이고 해기차 표는 `sea_effect_conditions.csv`, `sea_effect_hourly.csv`입니다.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]  # notebooks/<단계 폴더>/ 에서 실행
sys.path.insert(0, str(ROOT))

from src import config

In [2]:
import json
import warnings

import geopandas as gpd
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
N_WINTERS = len(config.PERIODS)

In [3]:
segments = gpd.read_file(config.PROCESSED_DIR / "segments.gpkg")
seg_feat = pd.read_parquet(config.PROCESSED_DIR / "segment_features.parquet").merge(
    pd.read_parquet(config.PROCESSED_DIR / "segment_b3.parquet"), on=["seg_id", "time"], how="left",
    validate="one_to_one")  # 03-2의 구간 B3 (밑그림 + 그날의 오차 크리깅)
stations = pd.read_parquet(config.PROCESSED_DIR / "stations.parquet").set_index("stn")
station_seg = pd.read_csv(config.PROCESSED_DIR / "station_segment_match.csv")
sf = pd.read_parquet(config.PROCESSED_DIR / "station_features.parquet").merge(
    pd.read_parquet(config.PROCESSED_DIR / "station_estimates.parquet"), on=["stn", "time"])
est_meta = json.load(open(config.MODEL_DIR / "road_temp_meta.json", encoding="utf-8"))
cls_meta = json.load(open(config.MODEL_DIR / "icing_meta.json", encoding="utf-8"))
print(f"구간 {len(segments)}개, 구간 시계열 {seg_feat.shape[0]:,}행 (1시간 간격)")

구간 233개, 구간 시계열 1,342,080행 (1시간 간격)


## 1. 구간별 관측값 추정

03의 노면온도 앙상블(LightGBM + Ridge + 주변 RWIS IDW + B3)과 04의 결빙 분류 모델을 구간 중간점 변수에 적용합니다.

In [4]:
def predict_members(df):
    """03의 최종 추정 방법(앙상블)의 구성원별 노면온도 추정값."""
    feats = est_meta["features"]
    booster = lgb.Booster(model_file=str(config.MODEL_DIR / "road_temp_lgbm.txt"))
    ridge = joblib.load(config.MODEL_DIR / "road_temp_ridge.joblib")
    return pd.DataFrame({"LightGBM(핵심)": booster.predict(df[feats]), "Ridge(핵심)": ridge.predict(df[feats]),
                         "B1 주변 노면온도": df["nbr_rst"].to_numpy(), est_meta["b3"]: df["b3"].to_numpy()}, index=df.index)


def predict_icing(df):
    """04의 최종 결빙 분류 모델로 결빙(결빙관련 노면) 확률을 구한다."""
    path = config.MODEL_DIR / cls_meta["model_file"]
    if cls_meta["model_type"] == "lightgbm":
        return lgb.Booster(model_file=str(path)).predict(df[cls_meta["features"]])
    return joblib.load(path).predict_proba(df[cls_meta["features"]])[:, 1]


members = predict_members(seg_feat)
seg_feat["rst_est"] = members[est_meta["ensemble"]].mean(axis=1).astype("float32")
seg_feat["rst_dpd"] = seg_feat["rst_est"] - seg_feat["td"]
seg_feat["p_icing"] = predict_icing(seg_feat).astype("float32")
seg_feat[["rst_est", "p_icing"]].describe().round(3)

,rst_est,p_icing
count,1342080.000,1342080.000
mean,4.498,0.025
std,6.518,0.113
min,-13.017,0.000
25%,-0.339,0.000
50%,3.742,0.001
75%,8.642,0.002
max,27.742,0.972


RWIS 관측소가 있는 구간에서 정시 관측값과 비교했습니다. 검증 지점 3곳은 학습에 쓰지 않았으므로 독립적인 확인이 됩니다.

In [5]:
obs = sf[sf["time"].dt.minute == 0][["stn", "time", "rst"]].merge(station_seg[["stn", "seg_id", "holdout"]], on="stn")
chk = obs.merge(seg_feat[["seg_id", "time", "rst_est"]], on=["seg_id", "time"]).dropna()
err = chk["rst_est"] - chk["rst"]
seg_check = (chk.assign(err=err).groupby(["stn", "seg_id", "holdout"])["err"]
             .agg(BIAS="mean", MAE=lambda e: e.abs().mean(), RMSE=lambda e: np.sqrt((e ** 2).mean())))
seg_check = seg_check.join(stations["name"], on="stn")
seg_check.round(2)

,,,BIAS,MAE,RMSE,name
stn,seg_id,holdout,,,,
12103,SH-099,False,0.18,0.45,0.67,군산휴(도)
12209,SH-070,False,0.73,0.80,1.16,서김제(도)
12210,SH-048,False,0.03,0.37,0.48,부안고려청자휴(도)
12211,SH-021,False,0.41,0.49,0.62,고인돌휴(도)
12409,SH-091,False,0.70,0.72,0.88,우곡3교(도)
12410,SH-044,True,-0.54,0.68,0.91,부곡교(도)
23202,HN-083,False,-0.33,0.39,0.51,여산휴(도)
23203,HN-067,False,-0.06,0.34,0.47,삼례졸음(도)
23204,HN-020,False,0.04,0.35,0.48,정읍휴(도)


학습 지점 구간은 자기 관측이 주변 RWIS 변수에 들어가 있어 오차가 작게 나옵니다. 검증 지점 구간은 부곡교 0.91, 해전 1.00, 삼봉 0.77°C로 03-3과 같습니다.

학습 지점 구간의 오차는 B3를 더하기 전(0.32~0.70°C)보다 커졌습니다(0.37~1.16°C). B3는 원천 관측소의 지문을 옮기지 않고(φ = 0) 그날의 오차도 너깃만큼 걸러서 옮기기 때문입니다. 휴게소 안 센서처럼 본선과 다른 자리의 버릇은 옮기지 않는 편이 맞습니다. 다만 서김제(0.70에서 1.16°C)와 우곡3교(0.54에서 0.88°C)처럼 본선 위 센서가 있는 구간에서는 추정이 관측값에서 멀어집니다.

구간 추정을 읽을 때 유의할 점은 다음과 같습니다(03-2의 3-4~3-6).

- 추정 오차의 절반 가까이는 지점마다 같은 시각에 반복되는 고유 편향입니다. 한낮에 특히 큽니다.
- 학습 지점 14곳 중 7곳이 휴게소·졸음쉼터 안의 센서입니다. 이 센서들은 본선 센서보다 한낮에 약 1.8°C, 밤에 0.3~0.4°C 따뜻합니다. 보정해 봤지만 검증 지점에서 나빠져 적용하지 않았습니다.
- 위험도는 구간 간 상대 비교이고 주로 밤·새벽을 보므로 이 편향의 영향은 한낮보다 작습니다.
- 편향의 방향이 일정하지 않습니다. 학습 지점의 본선 센서에서는 추정이 높은 경향이었고 검증 지점 3곳(모두 본선)에서는 평균 0.4°C 낮았습니다.
- B3는 0.65°C/100m 감률을 그대로 써서 주변보다 높은 곳의 밤 노면온도를 조금 낮게 추정합니다(03-2의 3-6-6). 산지 구간(통영-대전중부선)의 밤 저온 노출은 조금 크게 나올 수 있습니다.

B3를 더해 구간 추정이 얼마나 바뀌었는지 기존 앙상블(LightGBM + Ridge + B1)과 비교했습니다(값은 새 앙상블 - 기존 앙상블). 가장 가까운 학습 관측소까지 거리별로 봅니다.

In [6]:
_g = gpd.GeoSeries(gpd.points_from_xy(stations["lon"], stations["lat"]), crs=4326).to_crs(5179)
train_xy = np.c_[_g.x, _g.y][~stations["holdout"].to_numpy()]
seg_near = pd.Series(cdist(segments[["mid_x", "mid_y"]].to_numpy(), train_xy).min(axis=1) / 1000,
                     index=segments["seg_id"])  # 가장 가까운 학습 관측소까지 거리 (km)
diff = seg_feat["rst_est"] - members[["LightGBM(핵심)", "Ridge(핵심)", "B1 주변 노면온도"]].mean(axis=1)
by_seg = pd.DataFrame({"평균 차이": diff.groupby(seg_feat["seg_id"]).mean(),
                       "평균 |차이|": diff.abs().groupby(seg_feat["seg_id"]).mean(),
                       "밤 평균 차이": diff[seg_feat["sun_elev"] < 0].groupby(seg_feat["seg_id"]).mean(),
                       "한낮 평균 차이": diff[seg_feat["time"].dt.hour.between(11, 14)].groupby(seg_feat["seg_id"]).mean()})
band = pd.cut(seg_near.reindex(by_seg.index), [0, 2, 5, 10, 20], labels=["2km 이내", "2~5km", "5~10km", "10~20km"])
b3_effect = by_seg.groupby(band, observed=True).mean()
b3_effect.insert(0, "구간 수", band.value_counts())
b3_effect.round(2)

,구간 수,평균 차이,평균 |차이|,밤 평균 차이,한낮 평균 차이
2km 이내,58,0.01,0.21,-0.00,0.06
2~5km,73,0.00,0.20,-0.02,0.07
5~10km,63,0.07,0.16,0.05,0.16
10~20km,39,0.12,0.15,0.07,0.27


바뀐 크기(평균 |차이|)는 0.15~0.21°C이고 관측소에 가까운 구간에서 조금 큽니다. 가까운 구간일수록 B1이 그 관측소의 버릇을 많이 옮기는데 B3는 옮기지 않기 때문입니다. 관측소에서 먼 구간(10~20km)은 평균 0.12°C, 한낮 0.27°C 따뜻해졌습니다. B3는 구간의 일사, 지형 그늘, 운량으로 밑그림을 계산하므로 차이가 한낮에 커집니다.

결빙 판단에 쓰는 밤의 변화는 모든 거리대에서 0.1°C 미만입니다. B3를 더해도 05-2의 등급별 구간 수는 거의 바뀌지 않았습니다.

In [7]:
night = seg_feat["sun_elev"] < 0
seg_summary = seg_feat.groupby("seg_id").agg(
    rst_mean=("rst_est", "mean"),
    rst_min=("rst_est", "min"),
).join(seg_feat[night].groupby("seg_id")["rst_est"].mean().rename("rst_night_mean"))
seg_summary["rst_le0_hours"] = seg_feat.assign(c=seg_feat["rst_est"] <= 0).groupby("seg_id")["c"].sum() / N_WINTERS
seg_summary.round(2).describe()

,rst_mean,rst_min,rst_night_mean,rst_le0_hours
count,233.000000,233.000000,233.000000,233.000000
mean,4.498197,-10.499056,2.179614,773.667382
std,0.751830,0.695641,0.823552,131.145237
min,0.690000,-13.020000,-1.780000,660.000000
25%,4.260000,-10.790000,2.180000,694.000000
50%,4.810000,-10.600000,2.530000,721.000000
75%,4.990000,-10.150000,2.700000,774.500000
max,5.260000,-9.250000,2.830000,1476.500000


## 2. 결빙 영향인자 및 환경조건(임계값)

학습 지점 14곳의 10분 관측자료로 Ice가 관측된 때와 그렇지 않은 때의 조건을 비교합니다(검증 지점 제외). 노면온도는 실측값, 기상은 02-2의 보간값을 씁니다. 추가 자료로 ERA5-Land의 지난 3시간 강수·강설과 강설 비율, 서해 해기차도 같이 봅니다. ERA5는 약 9km 격자 값이라 지점의 실제 강수와 다를 수 있습니다.

결빙은 노면 위의 물이 어는점 이하의 노면에서 얼어 생깁니다. 물은 강수, 녹은 눈, 수증기 응결(서리) 세 경로로 공급됩니다(Zerr 1997, Hong et al. 2022에서 재인용). 이 지역 Ice의 81%는 강수 중이거나 강수 직후이고 노면센서는 1.5°C 미만에서만 결빙 계열 상태를 판정합니다(01-6의 13장). 그래서 환경조건을 저온과 수분 공급의 조합으로 구성했습니다.

In [8]:
tr = sf[~sf["stn"].isin(config.HOLDOUT_STATIONS) & sf["ice"].notna()].copy()
tr["rst_td"] = tr["rst"] - tr["td"]
era5 = pd.read_parquet(config.PROCESSED_DIR / "era5_10min.parquet",
                       columns=["stn", "time", "sf_mmh", "tp3h", "sf3h", "snowfrac3h"])
e5 = tr[["stn", "time"]].merge(era5, on=["stn", "time"], how="left", validate="one_to_one")
for c in ["sf_mmh", "tp3h", "sf3h", "snowfrac3h"]:
    tr[f"era5_{c}"] = e5[c].to_numpy()  # 02-3의 ERA5-Land (약 9km 격자)
sea = pd.read_parquet(config.PROCESSED_DIR / "sea_10min.parquet").set_index("time")
for c in ["sea_dT_c_A", "sea_t850_c_A", "sea_wind850_dir_deg_A"]:
    tr[c] = tr["time"].map(sea[c]).to_numpy()  # 02-3의 서해 해기차 (모든 지점에 같은 시계열)
FACTORS = {
    "rst": "노면온도(°C)", "ta": "기온(°C)", "ta_min12h": "12시간 최저기온(°C)", "td": "이슬점(°C)",
    "rh": "상대습도(%)", "rst_td": "노면온도-이슬점(°C)", "ws": "풍속(m/s)", "cloud": "전운량(1/10)",
    "rain_hrs12h": "최근 12시간 강수시간(h)", "hrs_since_rain": "강수 후 경과시간(h)", "snow": "적설(cm)",
    "sun_elev": "태양고도(°)",
    "era5_tp3h": "ERA5 3시간 강수(mm)", "era5_sf3h": "ERA5 3시간 강설(mm, 물 환산)", "era5_snowfrac3h": "ERA5 강설 비율(3시간)",
    "sea_dT_c_A": "서해 해기차(°C)", "sea_t850_c_A": "서해 850hPa 기온(°C)",
}
ice, non = tr[tr["ice"] == 1], tr[tr["ice"] == 0]
cond_table = pd.DataFrame({
    "변수": FACTORS.values(),
    "Ice P10": ice[list(FACTORS)].quantile(0.1).values,
    "Ice 중앙값": ice[list(FACTORS)].median().values,
    "Ice P90": ice[list(FACTORS)].quantile(0.9).values,
    "비결빙 중앙값": non[list(FACTORS)].median().values,
    "노면 0°C 이하 비결빙 중앙값": non.loc[non["rst"] <= 0, list(FACTORS)].median().values,
}, index=list(FACTORS))
auc = {}
for col in FACTORS:
    # 단일 변수로 Ice/비결빙을 가르는 힘 (AUC, 0.5=무관). 방향은 Ice 쪽이 낮으면 작을수록 위험
    x = tr[col]
    r = x.rank()
    n1, n0 = len(ice), len(non)
    a = (r[tr["ice"] == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    auc[col] = max(a, 1 - a)
cond_table["판별력(AUC)"] = pd.Series(auc)
cond_table["위험 방향"] = np.where(cond_table["Ice 중앙값"] < cond_table["비결빙 중앙값"], "낮을수록", "높을수록")
cond_table.sort_values("판별력(AUC)", ascending=False).round(2)

,변수,Ice P10,Ice 중앙값,Ice P90,비결빙 중앙값,노면 0°C 이하 비결빙 중앙값,판별력(AUC),위험 방향
era5_sf3h,"ERA5 3시간 강설(mm, 물 환산)",0.00,0.18,1.26,0.00,0.00,0.87,높을수록
hrs_since_rain,강수 후 경과시간(h),0.00,0.50,40.03,43.50,35.33,0.85,낮을수록
snow,적설(cm),0.00,2.43,13.00,0.00,0.00,0.85,높을수록
era5_tp3h,ERA5 3시간 강수(mm),0.00,0.20,1.39,0.00,0.00,0.83,높을수록
rain_hrs12h,최근 12시간 강수시간(h),0.00,1.89,5.20,0.00,0.00,0.83,높을수록
rst,노면온도(°C),-5.60,-1.70,1.02,3.40,-2.80,0.81,낮을수록
era5_snowfrac3h,ERA5 강설 비율(3시간),0.00,0.96,1.00,0.00,0.00,0.81,높을수록
sea_t850_c_A,서해 850hPa 기온(°C),-16.37,-11.02,-2.19,-3.79,-9.37,0.79,낮을수록
rst_td,노면온도-이슬점(°C),0.21,2.70,6.62,6.03,5.30,0.78,낮을수록
cloud,전운량(1/10),2.50,9.19,10.00,4.96,4.29,0.77,높을수록


노면온도와 기온이 낮고 습도와 운량이 높고 최근 강수·적설이 있을 때 Ice가 나옵니다. 노면이 0°C 이하인 비결빙 시간(오른쪽 열)과 비교하면 강수, 적설, 높은 습도에서 차이가 납니다.

단일 변수 판별력은 추가 자료인 ERA5 3시간 강설(물 환산)이 가장 높습니다(AUC 0.87). 다음은 강수 후 경과시간과 적설입니다(0.85). Ice 사례의 강설 비율 중앙값은 0.96이고 Ice의 절반 이상이 지난 3시간 ERA5 강수 0.1mm 이상입니다. ERA5 3시간 강수(0.83)는 AWS 최근 12시간 강수시간(0.83)과 같은 수준입니다. 서해 850hPa 기온(0.79)과 해기차(0.75)는 강설보다 낮습니다. 눈이 지금 내리는지보다 눈구름이 생기기 좋은 배경 조건을 나타내기 때문으로 보입니다.

임계값은 Ice 사례 분포로 정했습니다. 노면온도는 Ice 사례의 90%가 들어가는 상한(P90), 습도는 하한(P10)입니다. 수분 조건은 최근 12시간 강수, 적설, 노면온도 ≤ 이슬점(서리·응결) 중 하나입니다.

In [9]:
T_RST = float(np.round(ice["rst"].quantile(0.9) * 2) / 2)      # 0.5°C 단위
T_RH = float(np.floor(ice["rh"].quantile(0.1) / 5) * 5)       # 5% 단위


def icing_condition(df, rst_col="rst"):
    """결빙 가능 환경조건: 저온 + 고습 + 수분 공급."""
    moist = (df["rain_hrs12h"] > 0) | (df["snow"] > 0) | ((df[rst_col] - df["td"]) <= 0)
    return (df[rst_col] <= T_RST) & (df["rh"] >= T_RH) & moist


base_rate = tr["ice"].mean()
rows = []
for name, c in {
    f"노면온도 ≤ {T_RST}°C": tr["rst"] <= T_RST,
    f"+ 습도 ≥ {T_RH:.0f}%": (tr["rst"] <= T_RST) & (tr["rh"] >= T_RH),
    "+ 수분 공급 (= 결빙 가능 조건)": icing_condition(tr),
}.items():
    rows.append({"조건": name, "Ice 포착률": c[tr["ice"] == 1].mean(), "해당 시간 비율": c.mean(),
                 "조건 내 Ice 비율": tr.loc[c, "ice"].mean(), "향상도(lift)": tr.loc[c, "ice"].mean() / base_rate})
conditions = pd.DataFrame(rows).set_index("조건")
print(f"전체 Ice 비율 {base_rate:.5f}")
conditions.round(4)

전체 Ice 비율 0.00050


,Ice 포착률,해당 시간 비율,조건 내 Ice 비율,향상도(lift)
조건,,,,
노면온도 ≤ 1.0°C,0.8996,0.3533,0.0013,2.5465
+ 습도 ≥ 70%,0.8285,0.2456,0.0017,3.3728
+ 수분 공급 (= 결빙 가능 조건),0.7531,0.1279,0.0029,5.8873


결빙 가능 조건은 전체 시간의 일부만 차지하면서 Ice 사례의 대부분을 포함합니다. 자료로 정한 노면온도 임계값 1.0°C는 Bonanno et al.(2010)의 강수 시 기준과 같습니다(Hong et al. 2022에서 재인용). 강수가 없을 때의 0°C 기준은 쓰지 않았습니다. 강수 없는 Ice의 38%가 0~1.5°C에서 나왔기 때문입니다(01-6의 13-1). 서리 조건(노면 ≤ 이슬점)은 Ice로 판정된 경우가 드물어(01-6의 13-3) 보조 조건으로만 씁니다.

조건의 조합을 자료로 확인하려고 얕은 결정나무(깊이 3)도 학습했습니다.

In [10]:
tree_vars = ["rst", "rh", "rain_hrs12h", "snow", "rst_td", "cloud", "ws"]
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=500, class_weight="balanced", random_state=0)
tree.fit(tr[tree_vars].fillna(0), tr["ice"].astype(int))
print(export_text(tree, feature_names=[FACTORS.get(v, v) for v in tree_vars], decimals=1, show_weights=False))

|--- 적설(cm) <= 0.1
|   |--- 노면온도(°C) <= 1.6
|   |   |--- 상대습도(%) <= 88.6
|   |   |   |--- class: 0
|   |   |--- 상대습도(%) >  88.6
|   |   |   |--- class: 1
|   |--- 노면온도(°C) >  1.6
|   |   |--- 적설(cm) <= 0.0
|   |   |   |--- class: 0
|   |   |--- 적설(cm) >  0.0
|   |   |   |--- class: 0
|--- 적설(cm) >  0.1
|   |--- 최근 12시간 강수시간(h) <= 0.4
|   |   |--- 노면온도-이슬점(°C) <= 3.1
|   |   |   |--- class: 1
|   |   |--- 노면온도-이슬점(°C) >  3.1
|   |   |   |--- class: 0
|   |--- 최근 12시간 강수시간(h) >  0.4
|   |   |--- 노면온도(°C) <= 1.6
|   |   |   |--- class: 1
|   |   |--- 노면온도(°C) >  1.6
|   |   |   |--- class: 0



결정나무는 적설, 노면온도, 습도, 최근 강수, 노면-이슬점 차로 나눕니다. 저온과 수분 공급의 조합으로 가른다는 점에서 위의 결빙 가능 조건과 같은 방향입니다.

관측소 단위로 도로·지형 요인과 결빙 지표의 관계를 봤습니다(학습 지점 14곳, 순위상관). 지점 수가 적어서 요인의 방향을 확인하는 근거로만 씁니다.

기상 요인 후보로 ERA5 강설 중 추정 노면온도가 0°C 이하인 시간 비율(05-2의 M3와 같은 정의)도 넣고, 같은 지표를 AWS 강수 감지로 만든 것과 비교했습니다. 구간에는 실측 노면온도가 없으므로 둘 다 03의 추정 노면온도(학습 지점은 교차검증 예측)를 씁니다.

In [11]:
tr["era5_frz_snow"] = (tr["era5_sf_mmh"] > 0.05) & (tr["rst_est"] <= 0)  # ERA5 강설 0.05 mm/h 이상 & 추정 노면 0°C 이하
tr["aws_frz_rain"] = (tr["rain"] >= 0.5) & (tr["rst_est"] <= 0)         # 같은 지표를 AWS 강수 감지로
stn_stats = tr.groupby("stn").agg(
    Ice비율=("ice", "mean"),
    결빙관련비율=("state", lambda s: s.isin(cls_meta["label_states"]).mean()),
    노면0도이하비율=("rst", lambda s: (s <= 0).mean()),
    결빙가능조건비율=("rst", lambda s: icing_condition(tr.loc[s.index]).mean()),
    ERA5강설영하노면=("era5_frz_snow", "mean"),
    AWS강수영하노면=("aws_frz_rain", "mean"),
).join(stations[["name", "elev", "tpi", "shade_winter", "dist_bridge", "dist_river"]])
targets = ["Ice비율", "결빙관련비율", "노면0도이하비율", "결빙가능조건비율"]
STN_FACTORS = {"elev": "표고", "tpi": "TPI(지형 상대고도)", "shade_winter": "겨울 지형 그늘", "dist_bridge": "교량까지 거리",
               "dist_river": "하천까지 거리", "ERA5강설영하노면": "ERA5 강설 & 추정 노면 ≤ 0°C",
               "AWS강수영하노면": "AWS 강수 감지 & 추정 노면 ≤ 0°C"}
corr = pd.DataFrame({t: {f: spearmanr(stn_stats[f], stn_stats[t])[0] for f in STN_FACTORS} for t in targets})
corr.index = list(STN_FACTORS.values())
corr.round(2)

,Ice비율,결빙관련비율,노면0도이하비율,결빙가능조건비율
표고,-0.19,-0.21,0.68,0.72
TPI(지형 상대고도),0.27,0.02,-0.48,-0.59
겨울 지형 그늘,-0.34,0.21,0.44,0.48
교량까지 거리,0.38,0.48,-0.53,-0.17
하천까지 거리,-0.02,0.64,-0.53,-0.49
ERA5 강설 & 추정 노면 ≤ 0°C,0.31,0.49,0.14,0.27
AWS 강수 감지 & 추정 노면 ≤ 0°C,0.16,0.45,0.34,0.60


노면 0°C 이하 비율은 표고, 지형 그늘과 양(+)의 관계이고 교량·하천과의 거리와 음(-)의 관계입니다. 높고 그늘지고 교량·하천에 가까울수록 노면이 자주 차가워집니다. Ice 비율은 표고와 관계가 약합니다. 서해안선 저지대에 Ice가 많은데 강수·적설이 잦기 때문으로 보입니다. 그래서 위험도에 기상(수분 + 저온) 요인과 도로·지형(저온 노출) 요인을 같이 넣었습니다.

ERA5 강설 & 추정 노면 ≤ 0°C 비율은 Ice 비율과 0.31, 결빙관련 비율과 0.49의 순위상관을 보입니다. 같은 지표를 AWS 강수 감지로 만들면 0.16, 0.45입니다. 지점이 14곳뿐이라(5% 유의 수준에 |ρ| ≥ 0.54 필요) 차이를 확신할 수는 없습니다. 05-2에서는 이 지표(M3)를 기상 요인에 더합니다.

In [12]:
stn_stats.round(4)

,Ice비율,결빙관련비율,노면0도이하비율,결빙가능조건비율,ERA5강설영하노면,AWS강수영하노면,name,elev,tpi,shade_winter,dist_bridge,dist_river
stn,,,,,,,,,,,,
12103,0.0000,0.0189,0.2588,0.0867,0.0172,0.0078,군산휴(도),22.180799,-6.549400,0.1828,396.9076,449.6408
12209,0.0005,0.0137,0.3086,0.1125,0.0262,0.0134,서김제(도),20.164400,-0.000000,0.0000,29.8045,436.3778
12210,0.0034,0.0144,0.2381,0.0917,0.0360,0.0102,부안고려청자휴(도),20.164400,-0.038500,0.0015,534.6397,345.2879
12211,0.0001,0.0145,0.2936,0.1032,0.0352,0.0092,고인돌휴(도),60.493099,-35.660301,0.0306,977.9577,744.4217
12409,0.0002,0.0063,0.3174,0.1246,0.0162,0.0068,우곡3교(도),20.164400,-0.423500,0.0051,76.5769,212.9384
23202,0.0000,0.0030,0.2473,0.0982,0.0136,0.0057,여산휴(도),61.501301,-15.580800,0.0246,137.6151,158.5416
23203,0.0001,0.0074,0.2537,0.0892,0.0135,0.0071,삼례졸음(도),20.164400,-0.000000,0.0000,489.2331,482.5026
23204,0.0000,0.0145,0.2404,0.1216,0.0222,0.0119,정읍휴(도),20.164400,-3.510900,0.0055,1064.7175,964.2688
23403,0.0014,0.0222,0.2553,0.1361,0.0290,0.0190,호남터널(도),110.903999,-59.073799,0.0936,3776.6985,380.1644


서해 해기차(02-3의 6-4)는 모든 구간에 같은 시계열이라 구간끼리 가르지 못합니다. 어느 시각에 위험이 커지는지를 보는 시간 지표로 봤습니다. 서해에서 생긴 눈구름의 영향이라면 서해에 가까운 노선에서 강하고 내륙 산지에서 약해야 합니다. 노선별 판별력과 조건별 Ice 포착률을 봅니다.

조건은 기상청 예보기술 자료(손에 잡히는 예보기술 제50호, 3.3절)를 따릅니다. 해기차 임계값은 오대호 13°C, 서해 15°C이고, 850hPa 기온이 -8°C보다 높으면 비가 될 가능성이 커집니다. 서~북서풍(250~340°)은 찬 공기가 서해를 건너오는 풍향으로 이 분석에서 정했습니다.

In [13]:
def single_auc(y, x):
    """값이 클수록 Ice 쪽이면 1에 가까운 AUC (0.5=무관)."""
    y, r = np.asarray(y) == 1, pd.Series(np.asarray(x)).rank().to_numpy()
    n1, n0 = y.sum(), (~y).sum()
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


ROUTES = ["서해안", "호남선", "통영-대전중부선"]
route = tr["stn"].map(stations["route"])
sea_auc = pd.DataFrame({name: {"전체": single_auc(tr["ice"], tr[col]),
                               **{r: single_auc(tr.loc[route == r, "ice"], tr.loc[route == r, col]) for r in ROUTES}}
                        for col, name in {"sea_dT_c_A": "서해 해기차", "era5_sf3h": "ERA5 3시간 강설"}.items()}).T
dT, west = tr["sea_dT_c_A"], tr["sea_wind850_dir_deg_A"].between(250, 340)
SEA_RULES = {"해기차 ≥ 13°C (오대호 기준)": dT >= 13, "해기차 ≥ 15°C (서해 기준)": dT >= 15, "+ 서~북서풍": (dT >= 15) & west,
             "+ 850hPa ≤ -8°C (서해 강설 조건)": (dT >= 15) & west & (tr["sea_t850_c_A"] <= -8)}
once = ~tr["time"].duplicated()  # 해기차는 시각마다 값이 하나다
sea_rows = []
for name, c in SEA_RULES.items():
    row = {"조건": name, "겨울 시간 비율": c[once].mean()}
    for r in ROUTES:
        g = route == r
        row[f"{r} Ice 포착률"] = c[g & (tr["ice"] == 1)].mean()
        row[f"{r} 향상도"] = tr.loc[g & c, "ice"].mean() / tr.loc[g, "ice"].mean()
    sea_rows.append(row)
sea_cond = pd.DataFrame(sea_rows).set_index("조건")
print(sea_auc.round(3).to_string())
sea_cond.round(3)


                전체    서해안    호남선  통영-대전중부선
서해 해기차       0.752  0.807  0.730     0.581
ERA5 3시간 강설  0.873  0.850  0.936     0.906


,겨울 시간 비율,서해안 Ice 포착률,서해안 향상도,호남선 Ice 포착률,호남선 향상도,통영-대전중부선 Ice 포착률,통영-대전중부선 향상도
조건,,,,,,,
해기차 ≥ 13°C (오대호 기준),0.588,0.918,1.561,0.827,1.410,0.700,1.190
해기차 ≥ 15°C (서해 기준),0.476,0.884,1.859,0.808,1.695,0.700,1.471
+ 서~북서풍,0.434,0.871,2.004,0.788,1.815,0.700,1.611
+ 850hPa ≤ -8°C (서해 강설 조건),0.253,0.782,3.093,0.769,2.990,0.525,2.075


해기차의 판별력은 서해안선 0.81, 호남선 0.73, 통영-대전중부선 0.58로 서해에서 멀어질수록 약합니다. ERA5 3시간 강설은 세 노선 모두 0.85 이상입니다.

임계값만으로는 조건이 넓습니다. 해기차 13°C 이상은 겨울 시간의 59%, 15°C 이상은 48%입니다. 세 조건을 모두 쓰면(해기차 ≥ 15°C, 서~북서풍, 850hPa ≤ -8°C) 겨울 시간의 25%로 줄면서 서해안선 Ice의 78%, 호남선의 77%, 통영-대전중부선의 53%를 포함합니다. Ice가 나올 확률은 노선 평균의 3.1배, 3.0배, 2.1배입니다.

15°C는 칠발도 수온과 흑산도 850hPa 기온으로 정한 값이라 더 북쪽인 이 지점에 그대로 맞지는 않습니다. 그래서 판정 규칙으로 쓰지 않고 시간 지표로 남겼습니다(`sea_effect_hourly.csv`). 결빙 분류 입력에도 넣지 않았습니다. 04-2의 2-3에서 일별 F1 차이 구간이 0을 포함했고 Ice AP가 거의 그대로였습니다.

### 구간별 영향인자와 환경조건

앞의 표는 학습 지점 관측으로 본 영향인자입니다. 제출 형식의 구간별(1km) 표를 만들려고 같은 내용을 구간별 추정값으로 계산했습니다. 구간 233개마다 세 가지를 구합니다.

| 내용 | 계산 |
|------|------|
| 기상 요약 | 겨울철 평균 기온·습도·풍속, 강수 감지·적설·ERA5 강설 시간, 저온 시간 |
| 환경조건(임계값) | 모델이 결빙관련 노면으로 예측한 시간의 노면온도·기온·노면-이슬점 차 상한(P90)과 습도 하한(P10) |
| 상관분석 | 추정 노면온도 1.5°C 이하인 시간에서 결빙관련 확률과 기상요인의 순위상관(Spearman). 절댓값이 큰 순서로 주요 영향인자 1~3위를 적습니다 |

1.5°C는 노면센서가 결빙 계열 상태를 판정하는 경계입니다(01-6의 13-1). 결빙관련 확률은 이 기상요인으로 계산한 값이어서 상관은 모델이 구간마다 어떤 요인에 반응하는지를 나타냅니다. 관측과 맞는지는 학습 지점의 실제 결빙관련 노면으로 같은 계산을 해서 확인했습니다.

In [14]:
seg_wx = seg_feat[["seg_id", "time", "rst_est", "rst_dpd", "p_icing", "ta", "rh", "ws", "cloud", "rain", "rain_hrs12h",
                   "hrs_since_rain", "snow"]]
seg_era5_path = config.PROCESSED_DIR / "segment_era5_1h.parquet"  # 02-3: 구간이 쓰는 ERA5 격자를 모두 받았을 때만 있다
if seg_era5_path.exists():
    seg_wx = seg_wx.merge(pd.read_parquet(seg_era5_path, columns=["seg_id", "time", "sf_mmh"]), on=["seg_id", "time"],
                          how="left", validate="one_to_one")
COLD_RST = 1.5
SEG_FACTORS = {"rst_est": "노면온도", "ta": "기온", "rh": "습도", "rst_dpd": "노면-이슬점 차", "ws": "풍속", "cloud": "운량",
               "rain_hrs12h": "최근 12시간 강수시간", "hrs_since_rain": "강수 후 경과시간", "snow": "적설", "sf_mmh": "ERA5 강설"}
SEG_FACTORS = {k: v for k, v in SEG_FACTORS.items() if k in seg_wx}


def hours_per_winter(flag):
    return flag.groupby(seg_wx["seg_id"]).sum() / N_WINTERS


def rank_corr(df, target, cols):
    """target과 cols 각각의 순위상관."""
    return df[[target, *cols]].rank().corr()[target].drop(target)


weather = seg_wx.groupby("seg_id")[["ta", "rh", "ws"]].mean().set_axis(["평균 기온(°C)", "평균 습도(%)", "평균 풍속(m/s)"], axis=1)
weather["강수 감지 시간(h/겨울)"] = hours_per_winter(seg_wx["rain"] >= 0.5)
weather["적설 시간(h/겨울)"] = hours_per_winter(seg_wx["snow"] > 0)
if "sf_mmh" in seg_wx:
    weather["ERA5 강설 시간(h/겨울)"] = hours_per_winter(seg_wx["sf_mmh"] > 0.05)
weather["저온 시간(h/겨울)"] = hours_per_winter(seg_wx["rst_est"] <= COLD_RST)

pred = seg_wx[seg_wx["p_icing"] >= cls_meta["threshold"]].groupby("seg_id")
limits = pd.DataFrame({"결빙관련 예측 시간(h/겨울)": pred.size() / N_WINTERS, "노면온도 상한 P90(°C)": pred["rst_est"].quantile(0.9),
                       "기온 상한 P90(°C)": pred["ta"].quantile(0.9), "습도 하한 P10(%)": pred["rh"].quantile(0.1),
                       "노면-이슬점 차 상한 P90(°C)": pred["rst_dpd"].quantile(0.9)})

cold = seg_wx[seg_wx["rst_est"] <= COLD_RST]
seg_corr = pd.DataFrame({s: rank_corr(g, "p_icing", SEG_FACTORS) for s, g in cold.groupby("seg_id")}).T.rename(columns=SEG_FACTORS)
top3 = seg_corr.abs().apply(lambda r: pd.Series(r.nlargest(3).index, index=["주요 영향인자 1", "주요 영향인자 2", "주요 영향인자 3"]), axis=1)
seg_factors = (segments.set_index("seg_id")[["route"]].join(weather).join(limits).join(top3).join(seg_corr.add_prefix("상관: ")))
assert seg_factors.notna().all().all()

OBS_FACTORS = {"rst": "노면온도", "ta": "기온", "rh": "습도", "rst_td": "노면-이슬점 차", "ws": "풍속", "cloud": "운량",
               "rain_hrs12h": "최근 12시간 강수시간", "hrs_since_rain": "강수 후 경과시간", "snow": "적설", "era5_sf_mmh": "ERA5 강설"}
obs_cold = tr[tr["rst"] <= COLD_RST].assign(related=lambda d: d["state"].isin(cls_meta["label_states"]).astype(float))
obs_corr = pd.DataFrame({s: rank_corr(g, "related", OBS_FACTORS) for s, g in obs_cold.groupby("stn")}).T.rename(columns=OBS_FACTORS)
print("구간별 1위 영향인자:", seg_factors["주요 영향인자 1"].value_counts().to_dict())
print("\n구간별 환경조건 범위:")
print(limits.describe().loc[["min", "50%", "max"]].round(2).to_string())
by_route = seg_corr.groupby(seg_factors["route"]).mean().T
by_route.insert(0, "구간 평균 ρ (추정)", seg_corr.mean())
by_route.insert(1, "학습 지점 평균 ρ (관측)", obs_corr.mean())
by_route.round(2)

구간별 1위 영향인자: {'최근 12시간 강수시간': 87, '습도': 61, '노면-이슬점 차': 45, '적설': 27, '강수 후 경과시간': 13}

구간별 환경조건 범위:
     결빙관련 예측 시간(h/겨울)  노면온도 상한 P90(°C)  기온 상한 P90(°C)  습도 하한 P10(%)  노면-이슬점 차 상한 P90(°C)
min              10.5            -1.52          -1.91         66.94                 1.75
50%              28.0            -0.48          -1.03         84.42                 4.50
max              64.0             1.63           0.81         92.69                 7.61


route,구간 평균 ρ (추정),학습 지점 평균 ρ (관측),서해안,통영-대전중부선,호남선
노면온도,0.10,-0.05,0.01,0.26,0.12
기온,0.14,-0.00,-0.04,0.39,0.22
습도,0.55,0.16,0.60,0.54,0.49
노면-이슬점 차,-0.55,-0.17,-0.52,-0.63,-0.55
풍속,0.06,0.09,0.02,0.04,0.10
운량,0.32,0.20,0.29,0.39,0.32
최근 12시간 강수시간,0.62,0.28,0.62,0.64,0.62
강수 후 경과시간,-0.58,-0.26,-0.56,-0.58,-0.59
적설,0.58,0.27,0.61,0.55,0.55
ERA5 강설,0.41,0.24,0.40,0.49,0.37


구간별 1위 영향인자는 최근 12시간 강수시간 87개, 습도 61개, 노면-이슬점 차 45개, 적설 27개, 강수 후 경과시간 13개 구간입니다. 모두 수분 공급 쪽 요인이고 노면온도, 기온, 풍속이 1위인 구간은 없습니다. 노면이 이미 차가운 시간에는 더 차가운지보다 수분이 있는지가 결빙관련 노면을 가른다는 01-7의 13-9와 같은 결과입니다.

학습 지점의 실제 결빙관련 노면으로 계산하면 상관의 크기는 작지만(0.28 이하) 상위 세 인자가 같습니다(최근 12시간 강수시간, 적설, 강수 후 경과시간). 크기가 다른 이유는 확률은 연속값이고 관측은 0과 1뿐이며 센서 판정의 잡음이 있기 때문으로 보입니다.

노선별로는 기온과의 상관이 다릅니다(통영-대전중부선 0.39, 호남선 0.22, 서해안선 -0.04). 산지 노선은 맑고 추운 시간이 많아서 저온 시간 안에서는 눈이 오는 덜 추운 때에 결빙관련 확률이 높은 것으로 보입니다.

구간별 환경조건은 노면온도 상한 -1.5~1.6°C(중앙값 -0.5°C), 습도 하한 67~93%(중앙값 84%)입니다. 학습 지점 Ice로 정한 전체 기준(노면 ≤ 1.0°C, 습도 ≥ 70%)보다 좁은 이유는 모델 임계값 0.73을 넘는 시간만 보기 때문입니다.

## 저장

| 파일 | 내용 |
|------|------|
| `data/processed/segment_estimates.parquet` | 구간별 1시간 간격 추정 노면온도, 결빙관련 확률 (05-2에서 사용) |
| `data/processed/segment_summary.parquet` | 구간별 노면온도 요약 (05-2에서 사용) |
| `data/processed/icing_conditions.json` | 결빙 가능 조건의 임계값, 조건별 포착률과 향상도, 관측소 단위 순위상관 (05-2에서 사용) |
| `data/results/icing_factors.csv`, `icing_conditions.csv` | 결빙 영향인자와 환경조건(임계값), 학습 지점 기준 |
| `data/results/segment_factors.csv` | 구간별 기상 요약, 환경조건(임계값), 순위상관, 주요 영향인자 |
| `data/results/sea_effect_conditions.csv`, `sea_effect_hourly.csv` | 해기차 조건별 Ice 포착률과 시간별 지표 |

In [15]:
cond_table.round(3).to_csv(config.RESULT_DIR / "icing_factors.csv", encoding="utf-8-sig")
conditions.round(4).to_csv(config.RESULT_DIR / "icing_conditions.csv", encoding="utf-8-sig")
seg_factors.round(3).to_csv(config.RESULT_DIR / "segment_factors.csv", encoding="utf-8-sig")
sea_cond.round(4).to_csv(config.RESULT_DIR / "sea_effect_conditions.csv", encoding="utf-8-sig")
hourly = sea[sea.index.minute == 0]
hourly = pd.DataFrame({"해기차": hourly["sea_dT_c_A"], "850hPa 기온": hourly["sea_t850_c_A"], "850hPa 풍향": hourly["sea_wind850_dir_deg_A"],
                       "서해 강설 조건": (hourly["sea_dT_c_A"] >= 15) & hourly["sea_wind850_dir_deg_A"].between(250, 340)
                       & (hourly["sea_t850_c_A"] <= -8)})
hourly.round(2).to_csv(config.RESULT_DIR / "sea_effect_hourly.csv", encoding="utf-8-sig")

seg_feat[["seg_id", "time", "rst_est", "rst_dpd", "p_icing"]].to_parquet(
    config.PROCESSED_DIR / "segment_estimates.parquet", index=False)
seg_summary.to_parquet(config.PROCESSED_DIR / "segment_summary.parquet")
with open(config.PROCESSED_DIR / "icing_conditions.json", "w", encoding="utf-8") as f:
    json.dump({"T_RST": T_RST, "T_RH": T_RH, "conditions": conditions.to_dict(orient="index"),
               "station_corr": corr.to_dict()}, f, ensure_ascii=False, indent=2)
print("저장 완료")

저장 완료
